# Timelapse Analysis

Statistics from the timelapse data collected to identify date and time patterns, anomalies and outliers, for a better filtering and understanding of the data.

In [ ]:
import os

input_dir = "images"

try:
    subfolders = [
        f for f in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, f))
    ]
    print(f"📂 Found {len(subfolders)} subfolders: {subfolders}")
except FileNotFoundError:
    print(f"❌ Error: '{input_dir}' directory not found!")
    subfolders = []

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import re

timestamp_re = re.compile(r'(\d{8}_\d{6})')
# European timezone that auto-handles CET (UTC+1) / CEST (UTC+2)
local_tz = ZoneInfo("Europe/Vienna")

images_per_project = []

for subfolder in subfolders:
    # 📁 Setup folder paths
    input_folder_path = os.path.join(input_dir, subfolder)

    # 🖼️ Discover image files
    image_files = [f for f in os.listdir(input_folder_path) 
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    print(f"🖼️ Found {len(image_files)} images in '{subfolder}'")
    data_points = [timestamp_re.search(f).group(1) for f in image_files if timestamp_re.search(f)]
    # Cast to datetime objects and mark as Europe/Vienna local time.
    # ZoneInfo applies the correct UTC offset for CET (winter, +01:00)
    # and CEST (summer, +02:00), handling DST automatically.
    data_points = [datetime.strptime(ts, '%Y%m%d_%H%M%S').replace(tzinfo=local_tz) for ts in data_points]

    images_per_project.append({
        "project": subfolder,
        "image_count": len(image_files),
        "data_points": data_points
    })

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Flatten images_per_project into a tidy DataFrame
records = []
for proj in images_per_project:
    for dt in proj["data_points"]:
        records.append({
            "project": proj["project"],
            "datetime": dt,
            "date": dt.date(),
            "time": dt.time()
        })

df = pd.DataFrame(records)

# Create a numeric y-axis value: minutes since midnight
df["minutes"] = df["datetime"].dt.hour * 60 + df["datetime"].dt.minute

# Shared y-axis formatting constants
time_ticks = range(0, 24 * 60, 60)
time_labels = [f"{h:02d}:00" for h in range(24)]

# Get the sorted list of projects
projects = df["project"].unique()
n_projects = len(projects)

# Create subplots, one per project, stacked vertically
fig, axes = plt.subplots(n_projects, 1, figsize=(14, 2.5 * n_projects), sharex=True)

for ax, project in zip(axes, projects):
    proj_df = df[df["project"] == project]

    # Compute the mean and the ±WINDOW min window
    WINDOW = 45  # minutes
    mean_minutes = proj_df["minutes"].mean()
    in_window = (proj_df["minutes"] >= mean_minutes - WINDOW) & (proj_df["minutes"] <= mean_minutes + WINDOW)
    in_df, out_df = proj_df[in_window], proj_df[~in_window]

    # Connecting line in light gray so it doesn't dominate
    sns.lineplot(
        data=proj_df,
        x="date",
        y="minutes",
        marker="o",
        markersize=5,
        color="lightgray",
        ax=ax,
        legend=False
    )

    # Scatter: blue dots inside the window, gray dots outside
    ax.scatter(in_df["date"], in_df["minutes"], color="steelblue", s=30, zorder=5, label=f"Within ±{WINDOW} min")
    ax.scatter(out_df["date"], out_df["minutes"], color="gray", s=30, zorder=5, label=f"Outside ±{WINDOW} min")

    # Mean line in red
    ax.axhline(y=mean_minutes, color='red', linestyle='--', linewidth=1.5, alpha=0.8, label=f"Mean: {int(mean_minutes)//60:02d}:{int(mean_minutes)%60:02d}")

    ax.set_title(project)
    ax.set_ylabel("Time (HH:MM)")
    ax.set_yticks(time_ticks)
    ax.set_yticklabels(time_labels)
    ax.set_ylim(10*60, 16*60)  # small padding
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.legend(loc='upper right', fontsize=9)

# Only the bottom subplot gets the x-axis label
axes[-1].set_xlabel("Date")
plt.xticks(rotation=45)
fig.suptitle("Timelapse Image Capture Times by Project", fontsize=14)
plt.tight_layout()
plt.show()


## Statistics

Summary statistics derived from the timelapse data, including per-project totals, outlier counts (based on the ±WINDOW min window around the mean capture time), and the longest gaps without data.

In [ ]:
from datetime import timedelta
import numpy as np

WINDOW = 45  # minutes, must match the value used in the plotting cell

print("═" * 60)
print("📊  TIMELAPSE STATISTICS")
print("═" * 60)

grand_total = 0
all_data = []

for proj in images_per_project:
    name = proj["project"]
    dts = sorted(proj["data_points"])
    n = len(dts)
    grand_total += n

    # Date range
    first, last = dts[0], dts[-1]
    span = (last - first).days

    # Mean and window
    minutes_from_midnight = np.array([dt.hour * 60 + dt.minute for dt in dts])
    mean_min = minutes_from_midnight.mean()
    in_window = (minutes_from_midnight >= mean_min - WINDOW) & (minutes_from_midnight <= mean_min + WINDOW)
    n_outliers = int((~in_window).sum())
    pct_out = 100 * n_outliers / n

    # Longest gap
    gaps_seconds = [(dts[i + 1] - dts[i]).total_seconds() for i in range(n - 1)]
    max_gap_sec = max(gaps_seconds) if gaps_seconds else 0
    max_gap_str = str(timedelta(seconds=int(max_gap_sec)))

    # Average images per day
    daily_avg = n / max(span, 1)

    # Earliest / latest capture (mean-based window)
    earliest_hm  = f"{int(minutes_from_midnight.min()) // 60:02d}:{int(minutes_from_midnight.min()) % 60:02d}"
    latest_hm   = f"{int(minutes_from_midnight.max()) // 60:02d}:{int(minutes_from_midnight.max()) % 60:02d}"

    all_data.append((name, n, first, last, span, mean_min, n_outliers, pct_out, max_gap_str, daily_avg, earliest_hm, latest_hm))

    print(f"\n📁  {name}")
    print(f"   Images           : {n}")
    print(f"   Date range       : {first.date()} → {last.date()}  ({span} days)")
    print(f"   Avg images/day   : {daily_avg:.1f}")
    print(f"   Mean capture     : {int(mean_min) // 60:02d}:{int(mean_min) % 60:02d}")
    print(f"   Earliest / Latest: {earliest_hm} / {latest_hm}")
    print(f"   Outliers (±{WINDOW}min): {n_outliers} / {n}  ({pct_out:.1f}%)")
    print(f"   Longest gap      : {max_gap_str}")

print("\n" + "═" * 60)
print(f"🏁  TOTAL IMAGES: {grand_total}")
print("═" * 60)

# ─── DataFrame summary table ───
summary_df = pd.DataFrame(
    all_data,
    columns=[
        "Project", "Images", "First Date", "Last Date", "Days",
        "Mean (min)", "Outliers", "Outliers %", "Longest Gap",
        "Avg/Day", "Earliest", "Latest"
    ]
)
print("\n")
display(summary_df)
